# K-Means sobre TF-IDF

Agrupa las reseñas con K-Means sobre la representación TF-IDF reducida por SVD, en la corrida de referencia y sobre diez semillas.

## Rutas

Las rutas son relativas a la carpeta `notebooks/`. Los datos no forman parte del repositorio (ver README).

In [ ]:
# Comprobación de rutas
import os
PROY = os.path.abspath('..')
print("carpeta del notebook :", os.getcwd())
print("raiz del proyecto    :", PROY)
for _sub in ("Topic/Dataframes", "Topic/Resultados", "csv_parques"):
    _p = os.path.join(PROY, _sub)
    print(f"  {'OK ' if os.path.isdir(_p) else 'FALTA'}  {_sub}")
print("\nSi algo dice FALTA, el kernel no arranco en la carpeta del notebook,\n"
      "o los datos no estan donde se los espera (ver README).")

### Corrida de referencia (Tablas 4.3 y 4.4)

In [ ]:
#TFIDF KMEANS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import spacy
import warnings
import gc
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.metrics.pairwise import euclidean_distances
from nltk.corpus import stopwords
import nltk
from tqdm import tqdm
import os
import unicodedata
import pickle
os.environ["OMP_NUM_THREADS"] = "1"

warnings.filterwarnings("ignore", category=FutureWarning)
nltk.download('stopwords', quiet=True)


def remove_accents(text):
    if not isinstance(text, str):
        return text
    return ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )

# 1. CARGA Y VALIDACIÓN DE DATOS

df = pd.read_pickle("../df_generic_cleaned.pkl")

columna_texto_fuente = 'texto_limpio' if 'texto_limpio' in df.columns else 'text'
print(f"Datos cargados: {len(df)} reseñas.")
print(f"Columna fuente: {columna_texto_fuente}")

# 2. LEMATIZACIÓN CON SPACY

nlp = spacy.load("es_core_news_sm")
# Desactivar componentes innecesarios para acelerar (~3x más rápido)
nlp.disable_pipes(["ner", "parser"])

stop_words_base = set(stopwords.words('spanish'))


def lematizar_texto(texto):
    if not isinstance(texto, str) or not texto.strip():
        return ""
    texto = texto.lower()
    texto = re.sub(r'[^a-záéíóúñü\s]', '', texto)
    doc = nlp(texto)
    tokens_limpios = []
    for token in doc:
        lema = remove_accents(token.lemma_.lower())
        # Chequear stopwords contra el lema, no contra token.text
        if lema not in stop_words_base and len(lema) > 2 and not token.is_punct:
            tokens_limpios.append(lema)
    return " ".join(tokens_limpios)


print("Lematizando reseñas...")
tqdm.pandas(desc="Lematización")
df['texto_lematizado'] = df[columna_texto_fuente].progress_apply(lematizar_texto)

# Eliminar reseñas que quedaron vacías después de lematizar
n_vacias = (df['texto_lematizado'].str.strip() == '').sum()
print(f"Reseñas vacías post-lematización: {n_vacias}")
df = df[df['texto_lematizado'].str.strip() != ''].copy()

# Verificación: antes vs después
print("Ejemplos de lematización:")
for i in range(min(3, len(df))):
    print(f"\n  ANTES:   {df.iloc[i][columna_texto_fuente][:120]}")
    print(f"  DESPUÉS: {df.iloc[i]['texto_lematizado'][:120]}")

columna_texto = 'texto_lematizado'

# 3. STOPWORDS — Mismo que LDA

CONTEXTO = {
    'lugar', 'sitio', 'espacio', 'zona', 'parte', 'parque', 'montevideo', 'uruguay',
    'visita', 'visitar', 'paseo', 'pasear', 'ir', 'hacer', 'dia', 'noche',
    'fin', 'semana', 'año', 'vez', 'momento', 'cosa', 'caso', 'tema', 'tipo',
    'aunque', 'sino', 'pues', 'tal', 'tan', 'asi', 'ahi', 'alla', 'aca',
    'pais', 'ciudad', 'capital', 'gente', 'persona', 'publico', 'mate', 'pasar',
    'quedar', 'estar', 'ser', 'haber', 'tener', 'poder', 'deber',
    'dar', 'ver', 'saber', 'querer', 'llegar', 'decir', 'salir', 'venir',
    'encontrar', 'dejar', 'poner', 'parecer', 'llevar', 'seguir',
    'etc', 'siempre', 'alguno', 'solo', 'medio', 'ahora', 'mismo',
}

SENTIMIENTO = {
    'lindo', 'hermoso', 'bonito', 'bello', 'precioso',
    'feo', 'horrible', 'espantoso', 'terrible',
    'bueno', 'malo', 'buen',
    'excelente', 'espectacular', 'genial', 'fantastico', 'increible',
    'maravilloso', 'divino', 'impecable', 'impresionante',
    'pesimo', 'lamentable', 'desastre', 'patetico',
    'mejor', 'peor', 'mayor', 'menor', 'gran', 'grande',
    'bien', 'mal', 'super', 'muy', 'bastante', 'poco', 'mucho',
    'agradable', 'desagradable', 'recomendable', 'recomendado',
    'tranquilo', 'paz', 'relax',
    'gustar', 'encantar', 'disfrutar', 'recomendar',
    'valer', 'ideal', 'perfecto', 'destacar', 'pena', 'disfrutable',
}

ERRORES = {
    'lindar',
    'exelente', 'exelentes', 'escelente',
    'imprecionante', 'ermoso', 'ermosa',
    'recomeir', 'recomiendable', 'recomendacion',
    'mas',
}

CORTESIA = {'gracias'}

LUGARES = {'prado'} # aparece muy mencionado en lso comentarios por ser un barrio, asi que lo saco.


PROTEGER = set()   
nombres_parque = set()
for nombre in df['nombre_parque'].str.lower().unique():
    for token in nombre.split():
        nombres_parque.add(remove_accents(token))
nombres_parque -= PROTEGER

# Lista final (forma base) — LDA y TF-IDF
final_stopwords = {remove_accents(w) for w in stopwords.words('spanish')}
for grupo in (CONTEXTO, SENTIMIENTO, ERRORES, CORTESIA, LUGARES):
    final_stopwords |= {remove_accents(w) for w in grupo}
final_stopwords |= nombres_parque
final_stopwords = sorted(final_stopwords)

# NOTA: 'aire' y 'libre' quedan FUERA a propósito (señal de "aire libre").

# --- Print de control ---
print(f"Stopwords totales: {len(final_stopwords)}")
print(f"\nNombres de parque incluidos ({len(nombres_parque)}):\n{sorted(nombres_parque)}")
print(f"\nLista completa:\n{final_stopwords}")

# 4. VECTORIZACIÓN TF-IDF

print("\nVectorizando con TF-IDF...")
tfidf_vectorizer = TfidfVectorizer(
    stop_words=final_stopwords,
    min_df=10,
    max_df=0.6,
    ngram_range=(1, 2),
    sublinear_tf=True # esto hace que en lugar de usar la frecuencia bruta use 1 + log(tf) para reducir el impacto de palabras muy frecuentes en un documento
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df[columna_texto])
feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"Matriz TF-IDF: {tfidf_matrix.shape}")
print(f"Vocabulario: {len(feature_names)} términos")

# 5. REDUCCIÓN DE DIMENSIONALIDAD (SVD/LSA)

n_componentes_svd = 100

svd = TruncatedSVD(n_components=n_componentes_svd, random_state=42)
normalizer = Normalizer(copy=False)

tfidf_reducida = svd.fit_transform(tfidf_matrix)
tfidf_reducida = normalizer.fit_transform(tfidf_reducida)

varianza_explicada = svd.explained_variance_ratio_.sum()
print(f"Varianza explicada por {n_componentes_svd} componentes: {varianza_explicada:.2%}")

# 6. BÚSQUEDA DE K ÓPTIMO (Calinski-Harabasz + Davies-Bouldin + Inercia)

#rango_k = range(3, 10)
#resultados_busqueda = []

#print("BÚSQUEDA DE K ÓPTIMO")

#for k in rango_k:
    #kmeans_k = KMeans(
    #    n_clusters=k,
    #    init='k-means++',
    #    max_iter=300,
    #    n_init=5,
    #    random_state=42
    #)
    #abels_k = kmeans_k.fit_predict(tfidf_reducida)

    #inercia = kmeans_k.inertia_
    #ch = calinski_harabasz_score(tfidf_reducida, labels_k)
    #db = davies_bouldin_score(tfidf_reducida, labels_k)

    #resultados_busqueda.append({
    #    'k': k,
    #    'inertia': inercia,
    #    'calinski_harabasz': ch,
    #    'davies_bouldin': db,
    #})
    #print(f"  K={k:2d} | Inercia: {inercia:>10,.1f} | CH: {ch:>8,.1f} | DB: {db:.4f}")
    #del kmeans_k, labels_k
    #gc.collect()

# 6. BÚSQUEDA DE K ÓPTIMO (Calinski-Harabasz + Davies-Bouldin + Inercia)

rango_k = range(2, 10)
resultados_busqueda = []

print("BÚSQUEDA DE K ÓPTIMO")

for k in rango_k:
    # Agregamos n_init explícito y limitamos max_iter un poco para no quemar la RAM
    kmeans_k = KMeans(
        n_clusters=k,
        init='k-means++',
        max_iter=100,      # Bajamos de 300 a 100, es suficiente para convergencia
        n_init=5,
        random_state=42,
        algorithm='elkan'  # MUY IMPORTANTE EN MAC: 'elkan' es mucho más rápido y estable para matrices grandes
    )
    labels_k = kmeans_k.fit_predict(tfidf_reducida)

    inercia = kmeans_k.inertia_
    ch = calinski_harabasz_score(tfidf_reducida, labels_k)
    db = davies_bouldin_score(tfidf_reducida, labels_k)

    resultados_busqueda.append({
        'k': k,
        'inertia': inercia,
        'calinski_harabasz': ch,
        'davies_bouldin': db,
    })
    print(f"  K={k:2d} | Inercia: {inercia:>10,.1f} | CH: {ch:>8,.1f} | DB: {db:.4f}")
    
    # Liberar memoria agresivamente después de cada iteración
    del kmeans_k
    del labels_k
    import gc
    gc.collect()
df_busqueda = pd.DataFrame(resultados_busqueda)

# Gráfica de selección de K
fig, ax1 = plt.subplots(figsize=(10, 5))

color_ch = 'tab:blue'
ax1.set_xlabel('Número de clusters (K)', fontsize=12)
ax1.set_ylabel('Calinski-Harabasz (↑ mejor)', color=color_ch, fontsize=12)
ax1.plot(df_busqueda['k'], df_busqueda['calinski_harabasz'], 'o-', color=color_ch, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color_ch)

ax2 = ax1.twinx()
color_db = 'tab:red'
ax2.set_ylabel('Davies-Bouldin (↓ mejor)', color=color_db, fontsize=12)
ax2.plot(df_busqueda['k'], df_busqueda['davies_bouldin'], 's--', color=color_db, linewidth=2)
ax2.tick_params(axis='y', labelcolor=color_db)

ax1.axvline(x=6, color='gray', linestyle=':', alpha=0.7, label='K=6 (fijado)')
ax1.legend(loc='upper left')

plt.title('Selección de K: Calinski-Harabasz vs Davies-Bouldin', fontsize=14)
fig.tight_layout()
fig.savefig("../Topic/Resultados/Topico/TFIDF_seleccion_k_kmeans.png", dpi=300, bbox_inches="tight")
plt.show()

k_optimo_ch = int(df_busqueda.loc[df_busqueda['calinski_harabasz'].idxmax(), 'k'])
k_optimo_db = int(df_busqueda.loc[df_busqueda['davies_bouldin'].idxmin(), 'k'])
ch_en_6 = df_busqueda[df_busqueda['k'] == 6]['calinski_harabasz'].values[0]
db_en_6 = df_busqueda[df_busqueda['k'] == 6]['davies_bouldin'].values[0]

print(f"\nK óptimo por Calinski-Harabasz: {k_optimo_ch} (CH = {df_busqueda['calinski_harabasz'].max():,.1f})")
print(f"K óptimo por Davies-Bouldin: {k_optimo_db} (DB = {df_busqueda['davies_bouldin'].min():.4f})")
print(f"CH en K=6: {ch_en_6:,.1f}")
print(f"DB en K=6: {db_en_6:.4f}")

# 7. MODELO FINAL CON K=6

k_final = 6
print("Entrenando KMEANS)")

kmeans_model = KMeans(
    n_clusters=k_final,
    init='k-means++',
    max_iter=300,
    n_init=10,
    random_state=42
)

df['cluster_tema'] = kmeans_model.fit_predict(tfidf_reducida)

ch_final = calinski_harabasz_score(tfidf_reducida, df['cluster_tema'])
db_final = davies_bouldin_score(tfidf_reducida, df['cluster_tema'])
print(f"Inercia final: {kmeans_model.inertia_:,.1f}")
print(f"Calinski-Harabasz: {ch_final:,.1f}")
print(f"Davies-Bouldin: {db_final:.4f}")

# Distribución de reseñas por cluster
print(f"\nDistribución de reseñas por cluster:")
dist = df['cluster_tema'].value_counts().sort_index()
for cluster_id, count in dist.items():
    pct = count / len(df) * 100
    print(f"  Cluster {cluster_id}: {count:,} reseñas ({pct:.1f}%)")


# 8. PALABRAS CLAVE POR CLUSTER

print("PALABRAS CLAVE POR CLUSTER:")


def obtener_palabras_cluster(tfidf_matrix, labels, feature_names, n_top=15): #calcula el promedio de TF-IDF para cada palabra dentro de cada cluster y devuelve las palabras con mayor promedio como representativas del cluster
    
    clusters_palabras = {}
    for cluster_id in sorted(set(labels)):
        mask = labels == cluster_id
        cluster_mean = tfidf_matrix[mask].mean(axis=0).A1
        top_indices = cluster_mean.argsort()[::-1][:n_top]
        top_words = [(feature_names[i], cluster_mean[i]) for i in top_indices]
        clusters_palabras[cluster_id] = top_words

        keywords = ", ".join([w for w, s in top_words[:10]])
        print(f"  Cluster {cluster_id}: {keywords}")

    return clusters_palabras


clusters_palabras = obtener_palabras_cluster(
    tfidf_matrix, df['cluster_tema'].values, feature_names
)


def plot_top_words_kmeans(clusters_palabras, n_top=15, save_path=None):
    n_clusters = len(clusters_palabras)
    n_cols = 2
    n_rows = (n_clusters + 1) // 2
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows), sharex=False)
    axes = axes.flatten()

    for cluster_id, palabras in clusters_palabras.items():
        palabras_top = palabras[:n_top]
        words = [w for w, s in palabras_top]
        scores = [s for w, s in palabras_top]

        ax = axes[cluster_id]
        ax.barh(words, scores, height=0.7, color='teal')
        ax.set_title(f'Cluster {cluster_id}', fontdict={'fontsize': 14, 'fontweight': 'bold'})
        ax.invert_yaxis()
        ax.tick_params(axis='both', which='major', labelsize=10)

    for idx in range(n_clusters, len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle('Palabras clave por cluster (TF-IDF + KMeans)', fontsize=18, y=1.02)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


plot_top_words_kmeans(clusters_palabras, save_path="../Topic/Resultados/Topico/TFIDF_palabras_clave_clusters.png")


# 9. DISTRIBUCIÓN POR PARQUE

perfil_parques = pd.crosstab(
    df['nombre_parque'],
    df['cluster_tema'],
    normalize='index'
) * 100

print("\nDistribución de clusters por parque (%):")
print(perfil_parques.round(1))

fig, ax = plt.subplots(figsize=(12, 8))
perfil_parques.plot(
    kind='barh',
    stacked=True,
    ax=ax,
    colormap='tab10',
    edgecolor='white'
)
plt.title('Distribución de tópicos por parque (TF-IDF + KMeans)', fontsize=16)
plt.xlabel('% de reseñas', fontsize=12)
plt.ylabel('')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
fig.savefig("../Topic/Resultados/Topico/TFIDF_distribucion_clusters_por_parque.png", dpi=300, bbox_inches="tight")
plt.show()

# 10. INSPECCIÓN CUALITATIVA

print("\n" + "=" * 60)
print("RESEÑAS MÁS REPRESENTATIVAS POR CLUSTER")
print("=" * 60)

centroides = kmeans_model.cluster_centers_
n_reseñas_ejemplo = 5

for cluster_id in range(k_final):
    mask = df['cluster_tema'] == cluster_id
    indices_cluster = np.where(mask)[0]

    distancias = euclidean_distances(
        tfidf_reducida[indices_cluster],
        centroides[cluster_id].reshape(1, -1)
    ).flatten()

    top_indices_local = distancias.argsort()[:n_reseñas_ejemplo]
    top_indices_global = indices_cluster[top_indices_local]

    palabras_clave = ", ".join([w for w, s in clusters_palabras[cluster_id][:8]])
    print(f"\n{'─' * 60}")
    print(f"CLUSTER {cluster_id} — Palabras clave: {palabras_clave}")
    print(f"{'─' * 60}")

    for rank, idx in enumerate(top_indices_global, 1):
        texto = df.iloc[idx][columna_texto_fuente][:250]
        parque = df.iloc[idx]['nombre_parque']
        dist = distancias[top_indices_local[rank - 1]]
        print(f"  [{rank}] dist={dist:.3f} | {parque}")
        print(f"       \"{texto}...\"")

# ============================================================
# 11. TOP TF-IDF POR PARQUE
# ============================================================
print("\n" + "=" * 60)
print("TOP PALABRAS TF-IDF POR PARQUE")
print("=" * 60)


def obtener_top_tfidf_por_parque(df, tfidf_matrix, vectorizer, top_n=10):
    feature_names_local = vectorizer.get_feature_names_out()
    parques = df['nombre_parque'].unique()
    resultados = {}

    for parque in sorted(parques):
        mask = (df['nombre_parque'] == parque).values
        mean_tfidf = tfidf_matrix[mask].mean(axis=0).A1
        top_indices = mean_tfidf.argsort()[::-1][:top_n]
        sorted_tfidf = [(feature_names_local[i], mean_tfidf[i]) for i in top_indices]
        resultados[parque] = sorted_tfidf

        keywords = ", ".join([f"{word}" for word, score in sorted_tfidf])
        print(f"  {parque.upper()}: {keywords}")

    return resultados


top_keywords_parques = obtener_top_tfidf_por_parque(df, tfidf_matrix, tfidf_vectorizer)

df[['id', 'nombre_parque', 'text', 'cluster_tema']
].to_pickle("../Topic/Dataframes/df_tfidf_final.pkl")

# 12. RESUMEN DE MÉTRICAS

print("Resumen de Metricas y Configuración:")
print(f"Corpus: {len(df)} reseñas")
print(f"Preprocesamiento: lematización con spaCy (es_core_news_sm)")
print(f"Vocabulario post-filtrado: {len(feature_names)} términos")
print(f"Stopwords aplicadas: {len(final_stopwords)}")
print(f"Vectorización: TfidfVectorizer, min_df=10, max_df=0.6, ngram_range=(1,2), sublinear_tf=True")
print(f"Reducción de dimensionalidad: TruncatedSVD, {n_componentes_svd} componentes ({varianza_explicada:.2%} varianza)")
print(f"Clustering: KMeans, k-means++, max_iter=300, n_init=10")
print(f"Rango de K evaluado: {list(rango_k)}")
print(f"K óptimo por Calinski-Harabasz: {k_optimo_ch}")
print(f"K óptimo por Davies-Bouldin: {k_optimo_db}")
print(f"K utilizado: {k_final} (consistencia entre técnicas)")
print(f"Calinski-Harabasz en K={k_final}: {ch_final:,.1f}")
print(f"Davies-Bouldin en K={k_final}: {db_final:.4f}")
print(f"Inercia: {kmeans_model.inertia_:,.1f}")
print(f"Random state: 42")



analyzer = tfidf_vectorizer.build_analyzer()
topic_words_raw = [[w for w, s in clusters_palabras[c]][:100] for c in sorted(clusters_palabras.keys())]
ref_texts = [analyzer(t) for t in df[columna_texto].tolist()]
pickle.dump({'topic_words_raw': topic_words_raw, 'ref_texts': ref_texts, 'final_stopwords': final_stopwords},
            open("../Topic/Dataframes/ckpt_tfidf.pkl", "wb"))
print("ckpt_tfidf.pkl OK")

### Coherencia C_v de la corrida de referencia

In [ ]:
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

# Tokenizar el corpus (mismo texto lematizado)
textos_tokenizados = [texto.split() for texto in df[columna_texto].tolist()]

# Armar diccionario Gensim
diccionario = Dictionary(textos_tokenizados)

# Listas de palabras por cluster (clusters_palabras)
topic_words = []
for cluster_id in sorted(clusters_palabras.keys()):
    words = [w for w, score in clusters_palabras[cluster_id][:15]]
    topic_words.append(words)

# Calcular C_v
cm = CoherenceModel(
    topics=topic_words,
    texts=textos_tokenizados,
    dictionary=diccionario,
    coherence='c_v'
)

cv_score = cm.get_coherence()
print(f"Coherencia C_v (K-Means + TF-IDF): {cv_score:.4f}")

# Si querés ver coherencia por tópico individual:
cv_por_topico = cm.get_coherence_per_topic()
for i, cv_t in enumerate(cv_por_topico):
    print(f"  Cluster {i}: C_v = {cv_t:.4f}")

### Coherencia C_v sobre las palabras filtradas

In [ ]:
# === Coherencia C_v — K-Means + TF-IDF ===
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

TOP_N = 15
topic_words = [
    [w for w, score in clusters_palabras[c] if w not in final_stopwords][:TOP_N]
    for c in sorted(clusters_palabras.keys())
]

textos_tokenizados = [t.split() for t in df[columna_texto].tolist()]
diccionario = Dictionary(textos_tokenizados)

cm = CoherenceModel(topics=topic_words, texts=textos_tokenizados,
                    dictionary=diccionario, coherence='c_v')
print(f"Coherencia C_v (K-Means + TF-IDF): {cm.get_coherence():.4f}")
for i, s in enumerate(cm.get_coherence_per_topic()):
    print(f"  Cluster {i}: {s:.4f}")

### Diez semillas y barrido de K (Tablas 3.2, 4.1 y 4.2)

In [ ]:
# ============================================================
# MULTI-SEMILLA — K-Means + TF-IDF  (autocontenido)
# Barrido K=2..9 (criterio CH y DB) + top-100 en K=6, por semilla.
# Cada semilla mueve SVD y KMeans (todos los componentes estocasticos). Reusa el corpus
# lematizado cacheado por LDA (prep_lemmatizado.pkl); si no existe, lo construye.
# ============================================================
import pickle, os, re, unicodedata, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score
from nltk.corpus import stopwords
import nltk, spacy
from tqdm import tqdm
nltk.download('stopwords', quiet=True)

BD    = ".."
DFDIR = f"{BD}/Topic/Dataframes"
SEEDS = [42, 0, 1, 2, 3, 4, 5, 6, 7, 8]
K_RANGE = range(2, 10)
K_FINAL = 6
N_SVD = 100

def remove_accents(text):
    if not isinstance(text, str): return text
    return ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')

PREP = f"{DFDIR}/prep_lemmatizado.pkl"
if os.path.exists(PREP):
    d = pickle.load(open(PREP, 'rb')); corpus = d['corpus']; final_stopwords = d['final_stopwords']
    print(f"prep_lemmatizado.pkl cargado | {len(corpus)} docs")
else:
    df = pd.read_pickle(f"{BD}/df_generic_cleaned.pkl")
    col = 'texto_limpio' if 'texto_limpio' in df.columns else 'text'
    nlp = spacy.load("es_core_news_sm"); nlp.disable_pipes(["ner", "parser"])
    stop_base = set(stopwords.words('spanish'))
    def lem(t):
        if not isinstance(t, str) or not t.strip(): return ""
        t = re.sub(r'[^a-záéíóúñü\s]', '', t.lower())
        out = []
        for tok in nlp(t):
            l = remove_accents(tok.lemma_.lower())
            if l not in stop_base and len(l) > 2 and not tok.is_punct: out.append(l)
        return " ".join(out)
    tqdm.pandas(desc="Lematizando")
    lemm = df[col].progress_apply(lem)
    corpus = [x for x in lemm if x.strip()]
    CONTEXTO = {
        'lugar','sitio','espacio','zona','parte','parque','montevideo','uruguay',
        'visita','visitar','paseo','pasear','ir','hacer','dia','noche',
        'fin','semana','año','vez','momento','cosa','caso','tema','tipo',
        'aunque','sino','pues','tal','tan','asi','ahi','alla','aca',
        'pais','ciudad','capital','gente','persona','publico','mate','pasar',
        'quedar','estar','ser','haber','tener','poder','deber',
        'dar','ver','saber','querer','llegar','decir','salir','venir',
        'encontrar','dejar','poner','parecer','llevar','seguir',
        'etc','siempre','alguno','solo','medio','ahora','mismo'}
    SENTIMIENTO = {
        'lindo','hermoso','bonito','bello','precioso','feo','horrible','espantoso','terrible',
        'bueno','malo','buen','excelente','espectacular','genial','fantastico','increible',
        'maravilloso','divino','impecable','impresionante','pesimo','lamentable','desastre','patetico',
        'mejor','peor','mayor','menor','gran','grande','bien','mal','super','muy','bastante','poco','mucho',
        'agradable','desagradable','recomendable','recomendado','tranquilo','paz','relax',
        'gustar','encantar','disfrutar','recomendar','valer','ideal','perfecto','destacar','pena','disfrutable'}
    ERRORES = {'lindar','exelente','exelentes','escelente','imprecionante','ermoso','ermosa',
        'recomeir','recomiendable','recomendacion','mas'}
    CORTESIA = {'gracias'}; LUGARES = {'prado'}; PROTEGER = set()
    nombres = set()
    for nombre in df['nombre_parque'].str.lower().unique():
        for t in nombre.split(): nombres.add(remove_accents(t))
    nombres -= PROTEGER
    final_stopwords = {remove_accents(w) for w in stopwords.words('spanish')}
    for g in (CONTEXTO, SENTIMIENTO, ERRORES, CORTESIA, LUGARES):
        final_stopwords |= {remove_accents(w) for w in g}
    final_stopwords |= nombres
    final_stopwords = sorted(final_stopwords)
    pickle.dump({'corpus': corpus, 'final_stopwords': final_stopwords}, open(PREP, 'wb'))
    print(f"prep_lemmatizado.pkl creado | {len(corpus)} docs")

# --- vectorizacion TF-IDF (identica a la celda) ---
vec = TfidfVectorizer(stop_words=final_stopwords, min_df=10, max_df=0.6, ngram_range=(1, 2), sublinear_tf=True)
Xtfidf = vec.fit_transform(corpus)
fn = vec.get_feature_names_out()
analyzer = vec.build_analyzer()
ref_analyzer = [analyzer(doc) for doc in corpus]   # invariante a la semilla

def top_words(labels, n=100):
    out = []
    for c in sorted(set(labels)):
        mask = labels == c
        cm = Xtfidf[mask].mean(axis=0).A1
        out.append([fn[i] for i in cm.argsort()[::-1][:n]])
    return out

barrido = {}          # seed -> {k: (CH, DB)}
topw_by_seed = {}
for s in SEEDS:
    svd = TruncatedSVD(n_components=N_SVD, random_state=s)
    Xr = Normalizer(copy=False).fit_transform(svd.fit_transform(Xtfidf))
    curva = {}
    for k in K_RANGE:
        km = KMeans(n_clusters=k, init='k-means++', max_iter=100, n_init=5, random_state=s)
        lab = km.fit_predict(Xr)
        curva[k] = (float(calinski_harabasz_score(Xr, lab)), float(davies_bouldin_score(Xr, lab)))
        if k == K_FINAL:
            km6 = KMeans(n_clusters=K_FINAL, init='k-means++', max_iter=300, n_init=10, random_state=s)
            lab6 = km6.fit_predict(Xr)
            topw_by_seed[s] = top_words(lab6)
    barrido[s] = curva
    kbest = max(curva, key=lambda kk: curva[kk][0])
    print(f"TF-IDF seed={s:>2} | CH argmax=K{kbest} | " + " ".join(f"K{k}:CH={curva[k][0]:.0f}" for k in K_RANGE))

pickle.dump({'topic_words_raw_by_seed': topw_by_seed, 'ref_texts': ref_analyzer,
             'final_stopwords': final_stopwords, 'seeds': SEEDS},
            open(f"{DFDIR}/ckpt_tfidf_multiseed.pkl", "wb"))
pickle.dump({'barrido': barrido, 'criterio': 'CH,DB', 'k_range': list(K_RANGE), 'seeds': SEEDS},
            open(f"{DFDIR}/barrido_tfidf_multiseed.pkl", "wb"))
print("OK -> ckpt_tfidf_multiseed.pkl + barrido_tfidf_multiseed.pkl")